In [6]:
import numpy as np
from pyscf import gto, scf, lo, mp, cc

# ####  test H2 monomers ####
# a = 1.20577 # bond length in a cluster
# d = 100 # distance between each cluster
# unit = 'A' # unit of length
# na = 2 # size of a cluster (monomer)
# nc = 5 # set as integer multiple of monomers
# spin = 2 # spin per monomer
# elmt = 'O'
# basis = 'sto6g'
# atoms = ""
# for n in range(nc*na):
#     shift = ((n - n % na) // na) * (d-a)
#     atoms += f"{elmt} {n*a+shift:.5f} 0.00000 0.00000 \n"
# ###########################

# mol = gto.M(atom=atoms,
#             basis=basis,
#             verbose=4,
#             unit=unit,
#             symmetry=0,
#             charge=0,
#             spin=spin*nc,
#             max_memory=20000,
#             )

atomstring = '''
Fe -0.64257176050830 0.51803920561668 0.11515259480577
O 0.11174687498777 2.15027362958097 -0.78003307659188
H 0.70766775417542 2.71590643817051 -0.25346860209865
O -1.31193844940493 -0.30254437211926 -1.59153974321848
H -1.93648366806763 0.13093135988795 -2.20090760771949
O 0.02720918798052 1.33940572113505 1.82122676893196
H 0.65258765622944 0.90686870076571 2.43040311553976
O -1.39648053530566 -1.11444807905756 1.01023916292248
H -1.99331392681300 -1.67940484230818 0.48397490889195
O -2.28682023617329 1.63340828371306 0.41005372492049
H -2.35377021104600 2.47291065207153 -0.08301808819513
O 1.00194372931180 -0.59712335622834 -0.17950165163229
H 1.06807766676477 -1.43758842942053 0.31202545851205
H -3.18922098471738 1.27280247171927 0.48036417156888
H -0.66521991563992 -0.78352006296416 -2.14179468111308
H 1.90454646424149 -0.23678289755293 -0.24845043594753
H 0.43776650602076 2.17113603799071 -1.69794346797289
H -0.61875604533523 1.82123954705219 2.37159866149671
H -1.72126010670064 -1.13595000805262 1.92856878689935
'''

mol = gto.M(atom = atomstring,
            basis = {
                'default': 'sto6g',
                'Fe': 'sto6g'
                },
            verbose = 4,
            unit = 'angstrom',
            symmetry = 0,
            charge = 2,
            spin = 0,
            max_memory = 20000,
            )

mf = scf.UHF(mol).density_fit()
mf = mf.x2c()
# mf.chkfile = 'lsmf.chk'
# mf.init_guess = 'chk'
dm0 = mf.from_chk('../fehydro/lsmf.chk')
mf.max_cycle = 100
mf.level_shift = 0.5
mf.kernel(dm0=dm0)

stable = False
while not stable:
    print(f'mean-field stability test')
    if not stable:
        mo_i, _, stable,_ = mf.stability(return_status=True)
        dm = mf.make_rdm1(mo_i,mf.mo_occ)
        mf.kernel(dm0=dm)
    elif stable:
        print(f'UHF Energy: {mf.e_tot}, stability {stable}')
        break

System: uname_result(system='Linux', node='sharmagroup-rn', release='7.0.0-28-generic', version='#28~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Wed Jul  1 15:50:57 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Wed Jul 29 11:46:56 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 19
[INPUT] num. electrons = 84
[INPUT] charge = 2
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z  

In [ ]:
mymp = mp.MP2(mf)
mymp.set_frozen()
mymp.kernel()
efull_mp2 = mymp.e_corr
print(f'MP2 Corr = {efull_mp2:.8f}')

mycc = cc.CCSD(mf)
mycc.level_shift = 0.5
mycc.set_frozen()
mycc.kernel()
efull_ccsd = mycc.e_corr
print(f'CCSD Corr = {efull_ccsd:.8f}')

efull_t = mycc.ccsd_t()
efull_ccsd_t = efull_ccsd + efull_t
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}')


******** <class 'pyscf.mp.dfump2.DFUMP2'> ********
nocc = (np.int64(31), np.int64(31)), nmo = (49, 49)
frozen orbitals 11
max_memory 20000 MB (current use 330 MB)
E(DFUMP2) = -1718.64222375349  E_corr = -0.289186957576714
E(SCS-DFUMP2) = -1718.64536503877  E_corr = -0.292328242856791
E_corr(same-spin) = -0.0631108918099219
E_corr(oppo-spin) = -0.226076065766792
MP2 Corr = -0.28918696

******** <class 'pyscf.cc.dfuccsd.UCCSD'> ********
CC2 = 0
CCSD nocc = (np.int64(31), np.int64(31)), nmo = (49, 49)
frozen orbitals 11
max_cycle = 50
direct = 0
conv_tol = 1e-07
conv_tol_normt = 1e-06
diis_space = 6
diis_start_cycle = 0
diis_start_energy_diff = 1e+09
max_memory 20000 MB (current use 330 MB)
Init t2, MP2 energy = -0.28918695964968
Init E_corr(UCCSD) = -0.289186959701411
cycle = 1  E_corr(UCCSD) = -0.343106474634506  dE = -0.0539195149  norm(t1,t2) = 0.108138
cycle = 2  E_corr(UCCSD) = -0.362791368206336  dE = -0.0196848936  norm(t1,t2) = 0.04697
cycle = 3  E_corr(UCCSD) = -0.3741349188766

In [8]:
print(f'MP2 Corr = {efull_mp2:.8f}')
print(f'CCSD Corr = {efull_ccsd:.8f}')
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}')

MP2 Corr = -0.28918696
CCSD Corr = -0.37794048
CCSD(T) Corr = -0.38616224


In [10]:
from pyscf.data import elements
from pyscf import lo
from pyscf.lno.tools import autofrag_iao

def riao_localization(mf, lo_file=None):
    # IAO localization
    mol = mf.mol
    frozen = elements.chemcore(mol)
    orbocc = mf.mo_coeff[:,frozen:np.count_nonzero(mf.mo_occ)]
    lo_coeff = lo.iao.iao(mol, orbocc)
    lo_coeff = lo.orth.vec_lowdin(lo_coeff, mf.get_ovlp())
    moliao = lo.iao.reference_mol(mol)
    frag_lolist = autofrag_iao(moliao)
    if lo_file is not None:
        np.savez('./lo_coeff.npz', lo_coeff=lo_coeff)
    return lo_coeff, frag_lolist, moliao.elements

def uiao_localization(mf, lo_file=None):
    # IAO localization
    mol = mf.mol
    frozen = elements.chemcore(mol)
    orbocc_a = mf.mo_coeff[0][:,frozen:np.count_nonzero(mf.mo_occ[0])]
    orbocc_b = mf.mo_coeff[1][:,frozen:np.count_nonzero(mf.mo_occ[1])]
    lo_coeff_a = lo.iao.iao(mol, orbocc_a)
    lo_coeff_a = lo.orth.vec_lowdin(lo_coeff_a, mf.get_ovlp())
    lo_coeff_b = lo.iao.iao(mol, orbocc_b)
    lo_coeff_b = lo.orth.vec_lowdin(lo_coeff_b, mf.get_ovlp())
    lo_coeff = [lo_coeff_a, lo_coeff_b]
    moliao = lo.iao.reference_mol(mol)
    frag_lolist = autofrag_iao(moliao)
    frag_lolist = [[i,i] for i in frag_lolist]
    if lo_file is not None:
        np.savez('./lo_coeff.npz', lo_coeff_a=lo_coeff_a,lo_coeff_b=lo_coeff_b)
    return lo_coeff, frag_lolist, moliao.elements

def iao_localization(mf, lo_file=None):
    if isinstance(mf, scf.rhf.RHF):
        return riao_localization(mf, lo_file)
    elif isinstance(mf, scf.uhf.UHF):
        return uiao_localization(mf, lo_file)

In [ ]:
frozen = elements.chemcore(mol)
moa = mf.mo_coeff[0][:,frozen:np.count_nonzero(mf.mo_occ[0])]
mob = mf.mo_coeff[1][:,frozen:np.count_nonzero(mf.mo_occ[1])]
moc = np.hstack((moa, mob))

In [ ]:
moc = np.hstack((moa, mob))

(60, 62)


In [39]:
S = mf.get_ovlp()                      # AO overlap

moc = np.hstack((moa, mob))            # (nao, na+nb)

M = moc.T @ S @ moc                    # Gram matrix in the S metric
w, V = np.linalg.eigh(M)               # eigenvalues in [0, 2]

thresh = 1e-9                          # drop near-zero (redundant) directions
keep = w > thresh
mo_combined = moc @ V[:, keep] / np.sqrt(w[keep])   # broadcasts the 1/sqrt(w)

# sanity check: should be identity
# assert np.allclose(mo_combined.T @ S @ mo_combined, np.eye(mo_combined.shape[1]), atol=1e-10)

In [40]:
print(mo_combined.shape)
abs(mo_combined.T @ S @ mo_combined - np.eye(mo_combined.shape[1])).max()

(60, 58)


np.float64(2.1968261365579167e-07)

In [43]:
from afqmc.lno_afqmc import tools
s1e = mf.get_ovlp()
tools.mo_span(mo_combined, s1e, mob)

(np.float64(5.160460947450929e-11), np.float64(1.0000002182251426))